#### এই সপ্তাহের Math কেন দরকার

দুটো লেখা কতটা কাছাকাছি, তা কম্পিউটারকে বোঝাতে হলে আগে দুটো লেখাকে Vector বানাতে হয়, তারপর সেই Vector দুটোকে তুলনা করতে হয়। Dot product, Euclidean distance আর cosine similarity সেই তুলনার তিনটা মাপ। Embedding-এর তুলনা আর পরে RAG-এর খোঁজ, দুটোই এই মাপগুলোর ওপর দাঁড়িয়ে আছে।

#### Part 1: AI Theory

#### System Instructions ও User Messages

#### আগে একটা গল্প
রেস্তোরাঁর ম্যানেজার সকালেই ওয়েটারকে বলে রাখলেন, "সবার সাথে ভদ্রভাবে কথা বলবে, আর শুধু মেনুর খাবারই বিক্রি করবে।" তারপর একে একে গ্রাহকেরা এসে নিজের নিজের অর্ডার দিল। ম্যানেজারের কথা সারাদিন সব অর্ডারে খাটে, কিন্তু গ্রাহকের কথা শুধু তার নিজের টেবিলের জন্য। কোনো গ্রাহক যদি বলে, "মেনুতে নেই, তবু একটা বিরিয়ানি দাও", ওয়েটার ম্যানেজারের নিয়মই মানবে।

#### মূল ধারণা
LLM-কে যে লেখা পাঠানো হয়, তা কয়েকটা **message** দিয়ে তৈরি, আর প্রতিটা message-এর একটা **role** (ভূমিকা) থাকে।

**System message** হলো model-এর চলার নিয়ম, যেমন "তুমি একটা দোকানের সহকারী, ভদ্রভাবে ও সংক্ষেপে উত্তর দেবে।" যে মানুষ application বানায়, সে এটা লেখে। **User message** হলো ব্যবহারকারীর প্রশ্ন বা কথা। Model-এর দেওয়া উত্তরকে বলে **assistant message**।

দুটো আলাদা রাখা হয় কারণ নিয়ম আর প্রশ্ন মিশিয়ে লিখলে ব্যবহারকারীর কথা নিয়মকে ঢেকে দিতে পারে। আলাদা রাখলে model সাধারণত system-এর নির্দেশকে বেশি গুরুত্ব দেয়। তবে এটা একশো ভাগ নিশ্চয়তা নয়। কথোপকথনে system message সাধারণত শুরুতে একবার থাকে, তারপর user আর assistant message পালা করে আসে।

Python-এ এই message-গুলো সাধারণত dictionary-র একটা list হয়।

**Demo:** নিচের code-এ system আর user, দুই অংশ list-এ রেখে একটা লেখায় জোড়া লাগানো হয়েছে। এখানে কোনো LLM call হয় না, সত্যিকারের API call W05-এ আসবে।

#### মনে রাখার এক লাইন
> System message নিয়ম ঠিক করে, user message প্রশ্ন জানায়; দুটো আলাদা রাখলে model-কে নিয়ম মানানো সহজ হয়।

In [29]:
messages = [
    {"role": "system", "content": "তুমি একটা দোকানের সহকারী। ভদ্রভাবে ও সংক্ষেপে উত্তর দেবে।"},
    {"role": "user", "content": "আমার অর্ডার কখন আসবে?"},
]

for message in messages:
    print(f"{message['role']:7s}: {message['content']}")

print()
prompt = "\n".join(f"[{message['role']}] {message['content']}" for message in messages)
print(prompt)


system : তুমি একটা দোকানের সহকারী। ভদ্রভাবে ও সংক্ষেপে উত্তর দেবে।
user   : আমার অর্ডার কখন আসবে?

[system] তুমি একটা দোকানের সহকারী। ভদ্রভাবে ও সংক্ষেপে উত্তর দেবে।
[user] আমার অর্ডার কখন আসবে?


#### Prompt Structure ও Few-shot Prompting

#### আগে একটা গল্প
নতুন একজন কর্মচারীকে কাজ বোঝাতে ম্যানেজার চারটা কথা বললেন: "তুমি আমাদের ক্যাশিয়ার (ভূমিকা)। আজকের বিক্রির হিসাব মেলাবে (কাজ)। টাকা না মিললে আমাকে জানাবে, আর পাঁচটার আগে শেষ করবে (শর্ত)। হিসাবটা এক লাইনে লিখে দেবে (আউটপুটের ধরন)।" তারপর নিজে একদিন হিসাব মিলিয়ে দেখিয়ে দিলেন। শুধু কথা শুনে কর্মচারী যতটা বুঝল, নিজের চোখে একবার দেখে বুঝল তার চেয়ে অনেক বেশি।

#### মূল ধারণা
**Prompt** হলো model-কে দেওয়া লেখা। ভালো prompt-এ সাধারণত চারটা অংশ থাকে:

**ভূমিকা** বলে model কে সেজে কাজ করবে। **কাজ** বলে ঠিক কী করতে হবে। **শর্ত** বলে কী করা যাবে না, বা কতটা বড় হবে। **আউটপুটের ধরন** বলে উত্তর কেমন আকারে চাই, যেমন এক লাইনে, তালিকায় বা টেবিলে।

কোনো উদাহরণ না দিয়ে শুধু কাজ বললে তাকে বলে **zero-shot prompting**। কাজ বলার সাথে দুই-তিনটা উদাহরণ (প্রশ্ন আর ঠিক উত্তর) দেখিয়ে দিলে তাকে বলে **few-shot prompting**। Model উদাহরণের ধরন দেখে সেই ধরনেই উত্তর দেয়। উদাহরণ যত পরিষ্কার, উত্তর তত গোছানো হয়। উত্তরের আকার নিয়ে model বারবার ভুল করলে few-shot সবচেয়ে কাজে লাগে।

**Demo:** একই কাজের জন্য নিচের code-এ zero-shot আর few-shot prompt পাশাপাশি লেখা হিসেবে বানানো হয়েছে। কাজটা হলো ক্রেতার মন্তব্য ভালো না খারাপ, তা বলা।

#### মনে রাখার এক লাইন
> ভালো prompt-এ ভূমিকা, কাজ, শর্ত ও আউটপুটের ধরন থাকে; আর উদাহরণ দেখিয়ে দিলে (few-shot) model সেই ধরনেই উত্তর দেয়।

In [30]:
task = "নিচের মন্তব্যটা ভালো না খারাপ, এক শব্দে লেখো।"
comment = "খাবারটা ঠান্ডা ছিল আর দেরিতে এসেছে।"

zero_shot = f"""ভূমিকা: তুমি একজন মন্তব্য-বিশ্লেষক।
কাজ: {task}
মন্তব্য: {comment}
উত্তর:"""

examples = [
    ("খুব সুন্দর প্যাকেট, দ্রুত পেয়েছি।", "ভালো"),
    ("দাম বেশি আর মান খারাপ।", "খারাপ"),
]
example_text = "\n".join(f"মন্তব্য: {text}\nউত্তর: {answer}\n" for text, answer in examples)

few_shot = f"""ভূমিকা: তুমি একজন মন্তব্য-বিশ্লেষক।
কাজ: {task}

{example_text}
মন্তব্য: {comment}
উত্তর:"""

print("--- zero-shot ---")
print(zero_shot)
print()
print("--- few-shot ---")
print(few_shot)


--- zero-shot ---
ভূমিকা: তুমি একজন মন্তব্য-বিশ্লেষক।
কাজ: নিচের মন্তব্যটা ভালো না খারাপ, এক শব্দে লেখো।
মন্তব্য: খাবারটা ঠান্ডা ছিল আর দেরিতে এসেছে।
উত্তর:

--- few-shot ---
ভূমিকা: তুমি একজন মন্তব্য-বিশ্লেষক।
কাজ: নিচের মন্তব্যটা ভালো না খারাপ, এক শব্দে লেখো।

মন্তব্য: খুব সুন্দর প্যাকেট, দ্রুত পেয়েছি।
উত্তর: ভালো

মন্তব্য: দাম বেশি আর মান খারাপ।
উত্তর: খারাপ

মন্তব্য: খাবারটা ঠান্ডা ছিল আর দেরিতে এসেছে।
উত্তর:


#### Hallucination

#### আগে একটা গল্প
অচেনা এলাকায় রিকশাওয়ালাকে ঠিকানা জিজ্ঞেস করলে কেউ কেউ রাস্তা না জানলেও বলে দেয়, "ওই মোড়ের পর বাঁয়ে।" সে ঠকাতে চায় না, শুধু "জানি না" বলতে সংকোচ করে। ফল হলো, তুমি ভুল রাস্তায় অনেকটা দূর চলে যাও।

#### মূল ধারণা
**Hallucination** হলো model না জেনেও আত্মবিশ্বাসের সাথে ভুল বা বানানো তথ্য বলা।

কেন হয়? Week 01-এ দেখেছ, model পরের token অনুমান করে। যে লেখা শুনতে সবচেয়ে মানানসই, সে সেটাই লিখে যায়। সত্য যাচাই করার কোনো ধাপ তার ভেতরে নেই। তাই প্রশ্নের তথ্য তার শেখা data-তে না থাকলেও সে এমন উত্তর লেখে, যা পড়তে ঠিক মনে হয়।

কমানোর তিনটা উপায় আছে। প্রথমত, **নির্ভরযোগ্য তথ্য context-এ দেওয়া**: প্রশ্নের সাথে সঠিক লেখাটা দিয়ে বলা, "শুধু এই লেখা থেকে উত্তর দাও।" দ্বিতীয়ত, **শর্ত দেওয়া**: "উত্তর না জানলে 'জানি না' বলো।" তৃতীয়ত, **যাচাই করা**: সংখ্যা, তারিখ আর নাম নিজে বা code দিয়ে মিলিয়ে নেওয়া। এই তিন উপায় ঝুঁকি কমায়, কিন্তু একেবারে শূন্য করে না। পরে RAG-তে প্রথম উপায়টাই বড় করে কাজে লাগানো হবে।

**Demo:** নিচের code-এ তথ্য আর "জানি না" শর্ত দিয়ে একটা prompt বানানো হয়েছে। তথ্যটা বোঝার জন্য বানানো, আর প্রশ্নের উত্তর ওই তথ্যে নেই।

#### মনে রাখার এক লাইন
> Model সত্য যাচাই করে না, অনুমান করে; তাই নির্ভরযোগ্য তথ্য দাও, "জানি না" বলার অনুমতি দাও, আর উত্তর যাচাই করো।

In [31]:
fact = "আমাদের দোকান সকাল ৯টা থেকে রাত ৯টা পর্যন্ত খোলা থাকে। শুক্রবার বন্ধ।"
question = "দোকানে কি বিকাশে টাকা দেওয়া যায়?"

prompt = f"""শুধু নিচের তথ্য থেকে উত্তর দাও। তথ্যে উত্তর না থাকলে লেখো "জানি না"।

তথ্য: {fact}
প্রশ্ন: {question}"""

print(prompt)


শুধু নিচের তথ্য থেকে উত্তর দাও। তথ্যে উত্তর না থাকলে লেখো "জানি না"।

তথ্য: আমাদের দোকান সকাল ৯টা থেকে রাত ৯টা পর্যন্ত খোলা থাকে। শুক্রবার বন্ধ।
প্রশ্ন: দোকানে কি বিকাশে টাকা দেওয়া যায়?


#### Context Management

#### আগে একটা গল্প
সফরে যাওয়ার আগে ছোট একটা স্যুটকেসে জিনিস ভরতে হচ্ছে। জায়গা কম, তাই সবচেয়ে দরকারি জিনিসগুলোই থাকে, বাকিগুলো বাদ যায়। কিন্তু পাসপোর্ট যেকোনো অবস্থায় থাকবেই।

#### মূল ধারণা
Week 02-এ দেখেছ, context window সীমিত। কথোপকথন লম্বা হলে সব message একসাথে ঢোকে না। তখন ঠিক করতে হয়, কী রাখব আর কী বাদ দেব। একেই বলে **context management**।

চারটা সাধারণ উপায় আছে। **System message সবসময় রাখো**, কারণ সেটাই নিয়ম। **শেষের কয়েকটা message রাখো**, কারণ সাম্প্রতিক কথাই সাধারণত বেশি দরকারি। **পুরোনো অংশ বাদ না দিয়ে ছোট সারাংশ বানিয়ে রাখা** যায়। আর **দরকারি তথ্য**, যেমন ব্যবহারকারীর নাম বা অর্ডার নম্বর, আলাদা করে মনে রাখা যায়।

ঝুঁকি হলো, ছেঁটে ফেললে পুরোনো দরকারি কথাও হারিয়ে যেতে পারে। তাই কী বাদ যাচ্ছে, তা ভেবে ঠিক করতে হয়।

**Demo:** নিচের code-এ system message রেখে বাকি message-গুলো থেকে শেষের কয়েকটা রাখা হয়েছে। সীমা ধরা হয়েছে শব্দ গুনে, ১২ শব্দ। এটা শুধু বোঝার জন্য শব্দ গুনে করা, আসল token গুনে নয়। আসল token-এর সংখ্যা শব্দের চেয়ে সাধারণত বেশি হয়, বাংলায় আরও বেশি। কথোপকথনের লেখাগুলো বোঝার জন্য বানানো।

#### মনে রাখার এক লাইন
> সীমা পেরোলে system message রেখে শেষের message-গুলো রাখো, আর মনে রেখো যা ছাঁটা হলো তার ভেতরের তথ্য হারিয়ে যায়।

In [32]:
def keep_recent(messages, max_words):
    system = [m for m in messages if m["role"] == "system"]
    others = [m for m in messages if m["role"] != "system"]

    used = sum(len(m["content"].split()) for m in system)
    kept = []
    for message in reversed(others):
        words = len(message["content"].split())
        if used + words > max_words:
            break
        kept.append(message)
        used += words

    return system + list(reversed(kept)), used


chat = [
    {"role": "system", "content": "তুমি দোকানের সহকারী।"},
    {"role": "user", "content": "আমার নাম রফিক।"},
    {"role": "assistant", "content": "নমস্কার রফিক ভাই।"},
    {"role": "user", "content": "আমি একটা লাল শার্ট খুঁজছি।"},
    {"role": "assistant", "content": "আমাদের কাছে লাল শার্ট আছে।"},
    {"role": "user", "content": "দাম কত?"},
]

trimmed, used = keep_recent(chat, max_words=12)

print("আগে message:", len(chat), "| পরে message:", len(trimmed), "| শব্দ:", used)
for message in trimmed:
    print(f"{message['role']:9s}: {message['content']}")

assert all("রফিক" not in m["content"] for m in trimmed)


আগে message: 6 | পরে message: 3 | শব্দ: 10
system   : তুমি দোকানের সহকারী।
assistant: আমাদের কাছে লাল শার্ট আছে।
user     : দাম কত?


#### Basic Prompt Evaluation

#### আগে একটা গল্প
একজন শিক্ষক নতুন একটা পদ্ধতিতে পড়ালেন। পদ্ধতিটা ভালো কি না, তা কীভাবে বোঝা যাবে? ক্লাসের দু-একজনকে জিজ্ঞেস করে "মনে হয় ভালো" বললে হবে না। বরং দশটা প্রশ্নের একটা ক্লাস টেস্ট নিয়ে দেখতে হবে কতজন কতটা ঠিক উত্তর দিল।

#### মূল ধারণা
একটা prompt ভালো কি না, তা যাচাই করতে একটা ছোট পরীক্ষা বানানো হয়। কয়েকটা প্রশ্ন আর প্রতিটার **প্রত্যাশিত উত্তর** (ঠিক উত্তর কী হওয়া উচিত) একটা তালিকায় লেখা হয়। তারপর prompt চালিয়ে প্রতিটা উত্তর প্রত্যাশিত উত্তরের সাথে মেলানো হয়। মিললে **pass**, না মিললে **fail**। শেষে গুনে দেখা হয় কতটা pass হলো।

এতে সংখ্যা পাওয়া যায়, আর সংখ্যা দিয়ে দুটো prompt-এর তুলনা করা যায়। একটা-দুটো উত্তর পছন্দ হলেই prompt ভালো ধরে নেওয়া ঠিক নয়, কারণ ওই কয়েকটা হয়তো ভাগ্যে ঠিক হয়েছে।

**Demo:** নিচের code-এ চারটা প্রশ্ন আছে। model-এর উত্তরগুলো নকল, বোঝার জন্য হাতে বানানো। সেগুলো প্রত্যাশিত উত্তরের সাথে মিলিয়ে pass আর fail গোনা হয়েছে।

#### মনে রাখার এক লাইন
> প্রশ্ন আর প্রত্যাশিত উত্তরের তালিকা বানিয়ে model-এর উত্তর মেলাও, তারপর pass গুনে prompt-এর মান সংখ্যায় দেখো।

In [33]:
test_cases = [
    {"question": "২ + ৩ কত?", "expected": "৫", "model_answer": "৫"},
    {"question": "বাংলাদেশের রাজধানী কোথায়?", "expected": "ঢাকা", "model_answer": "ঢাকা"},
    {"question": "সপ্তাহে কয়দিন?", "expected": "সাত", "model_answer": "আট"},
    {"question": "জলের রং কেমন?", "expected": "রংহীন", "model_answer": "রংহীন, স্বচ্ছ"},
]

passed = 0
for case in test_cases:
    is_pass = case["expected"] in case["model_answer"]
    passed += is_pass
    print("PASS" if is_pass else "FAIL", "|", case["question"], "->", case["model_answer"])

print(f"মোট: {passed}/{len(test_cases)} pass")
assert passed == 3


PASS | ২ + ৩ কত? -> ৫
PASS | বাংলাদেশের রাজধানী কোথায়? -> ঢাকা
FAIL | সপ্তাহে কয়দিন? -> আট
PASS | জলের রং কেমন? -> রংহীন, স্বচ্ছ
মোট: 3/4 pass


#### LLM-এর Strength ও Limitations

#### আগে একটা গল্প
তোমার এক বন্ধু গল্প বলায় ওস্তাদ। যেকোনো ঘটনা সে সুন্দর করে গুছিয়ে বলতে পারে। কিন্তু দোকানের হিসাব মেলাতে বসলে সে প্রায়ই গুলিয়ে ফেলে। বুদ্ধিমান লোক হিসাব মেলাতে ক্যালকুলেটর ব্যবহার করে, আর বন্ধুর কাজ থাকে সেই ফল সুন্দর করে বলা।

#### মূল ধারণা
LLM লেখার কাজে ভালো: লেখা বানানো ও সাজানো, সারাংশ করা, এক ভাষা থেকে আরেক ভাষায় বদলানো, আর product-এর বর্ণনা লেখা। এসব কাজে একটু ভুল হলে মানুষ সহজেই ধরে ফেলতে পারে, আর ক্ষতিও কম।

কিন্তু order-এর bill-এর হিসাব, সংখ্যা ও তারিখের মতো জায়গায় LLM-কে একা ছেড়ে দেওয়া নিরাপদ নয়। Week 02-এ দেখেছ, সে হিসাব কষে না, পরের token অনুমান করে। ভুল সংখ্যা দেখতে ঠিক সংখ্যার মতোই লাগে, তাই যাচাই ছাড়া ধরা কঠিন।

মূল শিক্ষা হলো: **হিসাব code করবে, LLM শুধু সেই ফল ভাষায় সাজাবে।**

**Demo:** নিচের code-এ bill code দিয়ে হিসাব করা হয়েছে, তারপর ফলটা একটা বাক্যে সাজানো হয়েছে। বাক্য সাজানোর কাজটাই LLM করতে পারত। প্রতিটা পণ্যের সাথে (নাম, পরিমাণ, দাম) আছে, আর সংখ্যাগুলো বোঝার জন্য হাতে বানানো।

#### মনে রাখার এক লাইন
> হিসাব code করে, আর LLM সেই ফল সুন্দর করে ভাষায় সাজায়।

In [34]:
items = [("চাল", 2, 70), ("ডাল", 1, 120), ("তেল", 1, 180)]

total = sum(quantity * price for _, quantity, price in items)
print("code দিয়ে হিসাব:", total)

sentence = f"আপনার অর্ডারে {len(items)}টা পণ্য আছে, আর মোট বিল {total} টাকা।"
print(sentence)

assert total == 440


code দিয়ে হিসাব: 440
আপনার অর্ডারে 3টা পণ্য আছে, আর মোট বিল 440 টাকা।


#### Part 2: Math Track, Dot Product, Distance ও Cosine Similarity

#### Dot Product

#### আগে একটা গল্প
দোকানে তুমি ২টা কলম আর ৩টা খাতা কিনলে। কলমের দাম ৪ টাকা, খাতার দাম ১ টাকা। মোট কত লাগবে? প্রতিটা জিনিসের সংখ্যা তার দামের সাথে গুণ করে সব যোগ করতে হবে: 2×4 + 3×1 = 11। কেনার তালিকা [2, 3] আর দামের তালিকা [4, 1], আর এই "গুণ করে যোগ"-ই dot product।

#### মূল ধারণা
**Dot product** হলো দুটো Vector-এর একই জায়গার সংখ্যা গুণ করে সব গুণফল যোগ করা:

> a · b = a₁×b₁ + a₂×b₂ + … + aₙ×bₙ

উদাহরণ:

> [1, 2, 3] · [4, 5, 6] = 1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32
> [2, 3] · [4, 1] = 2×4 + 3×1 = 8 + 3 = 11

ফল একটা সংখ্যা, Vector নয়। দুই Vector-এর সংখ্যা যত বেশি জায়গায় একসাথে বড় হয়, ফল তত বড় হয়।

#### AI-র সাথে সম্পর্ক
Embedding-এর তুলনায় প্রথম মাপ হলো dot product: ফল যত বড়, দুটো Vector তত বেশি মেলে। Week 03-এর Attention-এ প্রতিটা শব্দের score এই dot product দিয়েই বের হয়েছিল, আর পরের cosine similarity-ও এর ওপর দাঁড়িয়ে।

**Demo:** নিচের code-এ দুটো উদাহরণই `np.dot` দিয়ে করা হয়েছে।

#### মনে রাখার এক লাইন
> Dot product হলো একই জায়গার সংখ্যা গুণ করে সব যোগ করা, আর ফল যত বড় মিল তত বেশি।

In [35]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
print(np.dot(a, b))

c = np.array([2, 3])
d = np.array([4, 1])
print(np.dot(c, d))

assert np.dot(a, b) == 32
assert np.dot(c, d) == 11


32
11


#### Euclidean Distance

#### আগে একটা গল্প
মাঠের একটা জায়গা থেকে আরেকটা জায়গায় যেতে হবে, কিন্তু রাস্তা ধরে ঘুরে নয়, কাক যেভাবে সোজা উড়ে যায় সেভাবে। সেই সোজা পথের দৈর্ঘ্যই দুই জায়গার দূরত্ব। পূর্বে ৩ পা আর উত্তরে ৪ পা সরলে সোজা দূরত্ব ৫ পা, Week 02-এ যেমন দেখেছ।

#### মূল ধারণা
**Euclidean distance** হলো দুটো বিন্দুর মধ্যে সোজা দূরত্ব। দুই বিন্দু (x₁, y₁) আর (x₂, y₂) হলে:

> d = √[(x₂ − x₁)² + (y₂ − y₁)²]

উদাহরণ:

> (1, 2) থেকে (4, 6): d = √[(4 − 1)² + (6 − 2)²] = √(9 + 16) = √25 = 5
> (1, 1) থেকে (4, 5): d = √[(4 − 1)² + (5 − 1)²] = √(9 + 16) = √25 = 5

Vector-এ সংখ্যা বেশি থাকলেও একই নিয়ম: প্রতিটা জায়গার ফারাকের বর্গ যোগ করে বর্গমূল নাও। NumPy-তে লেখা হয়:

> `np.sqrt(np.sum((a - b) ** 2))`

ক্রমটা মনে রাখো: আগে বিয়োগ, তারপর বর্গ, তারপর যোগ, সবার শেষে বর্গমূল। আরেকভাবে বললে, `a - b` Vector-টার magnitude-ই (Week 02) দুই বিন্দুর দূরত্ব।

#### AI-র সাথে সম্পর্ক
দুটো লেখার embedding-এর দূরত্ব যত কম, লেখা দুটোর অর্থ তত কাছাকাছি।

**Demo ১:** নিচের code-এ (1, 2) থেকে (4, 6)-এর দূরত্ব বের করা হয়েছে।

#### মনে রাখার এক লাইন
> Euclidean distance হলো দুই বিন্দুর সোজা দূরত্ব, আর দূরত্ব যত কম, দুটো তত কাছাকাছি।

In [36]:
a = np.array([1, 2])
b = np.array([4, 6])

print(np.sqrt(np.sum((a - b) ** 2)))

assert np.sqrt(np.sum((a - b) ** 2)) == 5.0


5.0


**Demo ২:** নিচের cell-এ (1, 1) থেকে (4, 5)-এর দূরত্ব দুইবার লেখা হয়েছে, দুটোই 5.0। শেষ দুই লাইনে দেখা যায়, `a - b`-র যোগফল −7 আর তার বর্গ 49। ভুল ক্রমে যোগ আগে করলে এই সংখ্যাই আসে, যা পরের "আমার ভুল ও মনে রাখার কথা" অংশে আলোচনা করা হয়েছে।

In [37]:
import numpy as np

a = np.array([1, 1])
b = np.array([4, 5])

dist = np.sqrt(np.sum((a - b)**2))

print(dist)
print(np.sqrt(np.sum((a - b)**2)))
print(np.sum(a-b))
print(np.sum(a-b)**2)

5.0
5.0
-7
49


#### Cosine Similarity

#### আগে একটা গল্প
দুজন মানুষ একই দিকে আঙুল তুলল। একজনের হাত লম্বা, আরেকজনের ছোট। তবু দুজনের দিক একই। আবার একজন পূর্বে আর অন্যজন উত্তরে আঙুল তুললে দিক একেবারে আলাদা। অনেক সময় আমাদের জানা দরকার হয় কে কোন দিকে মুখ করে আছে, কার হাত কতটা লম্বা তা নয়।

#### মূল ধারণা
**Cosine similarity** দুটো Vector-এর দিকের মিল মাপে। Week 03-এ দেখেছ, dot product-এর দুর্বলতা হলো Vector বড় হলে ফল এমনিতেই বড় হয়। তাই dot product-কে দুই Vector-এর দৈর্ঘ্য (magnitude) দিয়ে ভাগ করা হয়:

> cosine(a, b) = (a · b) ÷ (‖a‖ × ‖b‖)

এখানে ‖a‖ মানে a-র দৈর্ঘ্য, অর্থাৎ √(a₁² + a₂² + … + aₙ²)। এর মান সবসময় −১ থেকে ১-এর মধ্যে থাকে: ১ মানে একই দিক, ০ মানে কোনো সম্পর্ক নেই, −১ মানে উল্টো দিক।

উদাহরণ:

> [1, 2] আর [3, 4]: a · b = 11, ‖a‖ = √5 ≈ 2.236, ‖b‖ = 5, তাই cosine = 11 ÷ (2.236 × 5) ≈ 0.98387
> [1, 0] আর [0, 1]: a · b = 0, তাই cosine = 0.0

#### AI-র সাথে সম্পর্ক
Embedding-এর তুলনায় সাধারণত cosine similarity-ই ব্যবহার হয়। RAG-এ প্রশ্নের embedding আর প্রতিটা লেখার embedding-এর cosine বের করে সবচেয়ে বেশি মানের লেখাটাই খুঁজে আনা হয়। তাই এটাই RAG-এর ভিত্তি।

**Demo ১:** নিচের cell-এ [1, 2] আর [3, 4]-এর cosine ধাপে ধাপে বের করা হয়েছে: dot product, দুটোর দৈর্ঘ্য, তারপর ভাগ।

#### মনে রাখার এক লাইন
> Cosine similarity হলো dot product-কে দুই দৈর্ঘ্যের গুণফল দিয়ে ভাগ করা, আর এটা শুধু দিকের মিল মাপে।

In [38]:
a = np.array([1, 2])
b = np.array([3, 4])

S= np.dot(a, b)

a_vec = np.sqrt(np.sum((a)**2))
b_vec = np.sqrt(np.sum((b)**2))

cosine = S/(a_vec*b_vec)

print(S)
print(a_vec)
print(b_vec)
print(cosine)

11
2.23606797749979
5.0
0.9838699100999074


**Demo ২:** নিচের code-এ একটা `cosine` function বানানো হয়েছে, আর [1, 0] ও [0, 1]-এ চালানো হয়েছে। দুটো Vector পরস্পর সমকোণে, তাই মান 0.0।

In [39]:
def cosine(u, v):
    return np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))


print(cosine(np.array([1, 0]), np.array([0, 1])))
print(round(cosine(np.array([1, 2]), np.array([3, 4])), 5))

assert cosine(np.array([1, 0]), np.array([0, 1])) == 0.0


0.0
0.98387


**Demo ৩:** Week 03-এ দেখা গিয়েছিল, keyword search "গাড়ি ঠিক করার দোকান" প্রশ্নে ভুল লেখাটা বেছে নেয়, কারণ ঠিক উত্তরের সাথে একটা শব্দও মেলে না। এবার একই প্রশ্ন cosine similarity দিয়ে দেখা যাক। প্রশ্ন আর দুটো লেখার Vector হাতে বানানো, তিন সংখ্যা হলো গাড়ি-ভাব, মেরামত-ভাব ও কেনাবেচা-ভাব। এগুলো বোঝার জন্য হাতে বানানো সংখ্যা।

In [40]:
query = "গাড়ি ঠিক করার দোকান"
query_vec = np.array([0.9, 0.8, 0.1])

docs = {
    "মোটর গ্যারেজে যন্ত্র সারানো হয়": np.array([0.8, 0.9, 0.0]),
    "গাড়ি কেনাবেচার দোকান": np.array([0.9, 0.0, 0.9]),
}

print("Keyword search (মিলে যাওয়া শব্দ গুনে):")
for text in docs:
    common = set(query.split()) & set(text.split())
    print(" ", len(common), "টা শব্দ মিলেছে:", text)

print("Semantic search (cosine similarity):")
for text, vec in docs.items():
    print(" ", round(cosine(query_vec, vec), 3), text)

best_keyword = max(docs, key=lambda text: len(set(query.split()) & set(text.split())))
best_semantic = max(docs, key=lambda text: cosine(query_vec, docs[text]))
print("Keyword search বেছেছে :", best_keyword)
print("Semantic search বেছেছে:", best_semantic)

assert best_keyword == "গাড়ি কেনাবেচার দোকান"
assert best_semantic == "মোটর গ্যারেজে যন্ত্র সারানো হয়"


Keyword search (মিলে যাওয়া শব্দ গুনে):
  0 টা শব্দ মিলেছে: মোটর গ্যারেজে যন্ত্র সারানো হয়
  2 টা শব্দ মিলেছে: গাড়ি কেনাবেচার দোকান
Semantic search (cosine similarity):
  0.99 মোটর গ্যারেজে যন্ত্র সারানো হয়
  0.585 গাড়ি কেনাবেচার দোকান
Keyword search বেছেছে : গাড়ি কেনাবেচার দোকান
Semantic search বেছেছে: মোটর গ্যারেজে যন্ত্র সারানো হয়


দেখা যাচ্ছে, keyword search ভুল লেখাটা বেছেছে, কারণ তাতে দুটো শব্দ মেলে। Semantic search ঠিক লেখাটা বেছেছে, কারণ তার Vector প্রশ্নের Vector-এর দিকে সবচেয়ে বেশি মেলে (cosine প্রায় 0.99, অন্যটার প্রায় 0.59)।

#### Normalization ও Magnitude

#### আগে একটা গল্প
মানচিত্রে দুটো তীর আঁকা আছে, একটা লম্বা আর একটা ছোট, কিন্তু দুটো একই দিকে। তুমি শুধু দিক জানতে চাও। তখন দুটো তীরকেই ঠিক ১ সেন্টিমিটার লম্বা করে নিলে লম্বা-ছোটর ঝামেলা থাকে না, আর দিক একই থেকে যায়।

#### মূল ধারণা
Week 02-এ শিখেছ, Vector-এর **magnitude** (দৈর্ঘ্য) হলো:

> ‖a‖ = √(a₁² + a₂² + … + aₙ²)

NumPy-তে `np.linalg.norm(a)` দিয়ে এটা পাওয়া যায়।

যে Vector-এর দৈর্ঘ্য ঠিক ১, তাকে বলে **unit vector**। যেকোনো Vector-কে তার নিজের দৈর্ঘ্য দিয়ে ভাগ করলে সেটা unit vector হয়, আর দিক বদলায় না। একে বলে **normalization**। যেমন [3, 4]-এর দৈর্ঘ্য 5, তাকে 5 দিয়ে ভাগ করলে:

> [3, 4] ÷ 5 = [0.6, 0.8]
> নতুন দৈর্ঘ্য: √(0.6² + 0.8²) = √(0.36 + 0.64) = √1 = 1

#### AI-র সাথে সম্পর্ক
Cosine similarity-র ভাগটা আসলে দুটো Vector-কে normalize করে তারপর dot product নেওয়া। তাই অনেক embedding আগে থেকেই normalize করা থাকে, আর তখন শুধু dot product-ই cosine similarity-র সমান হয়।

**Demo ১:** নিচের code-এ [3, 4]-কে normalize করা হয়েছে।

#### মনে রাখার এক লাইন
> Normalization মানে Vector-কে তার দৈর্ঘ্য দিয়ে ভাগ করা, তাতে দৈর্ঘ্য হয় ১ আর দিক একই থাকে।

In [41]:
v = np.array([3, 4])

length = np.linalg.norm(v)
unit_v = v / length

print(length)
print(unit_v)
print(np.linalg.norm(unit_v))

assert length == 5.0
assert np.allclose(unit_v, [0.6, 0.8])
assert np.isclose(np.linalg.norm(unit_v), 1.0)


5.0
[0.6 0.8]
1.0


**Demo ২:** নিচের cell-এ [6, 8]-কে normalize করা হয়েছে। দৈর্ঘ্য 10, আর ফল আবার [0.6, 0.8]। এটা অবাক হওয়ার মতো নয়, কারণ [6, 8] হলো [3, 4]-এর দ্বিগুণ, অর্থাৎ দুটোর দিক একই।

In [42]:
a = np.array([6, 8])

mag = np.linalg.norm(a)
unit = a / mag

print(mag)
print(unit)
print(np.linalg.norm(unit))

10.0
[0.6 0.8]
1.0


#### আমার ভুল ও মনে রাখার কথা

| আমার ভুল | ঠিক | মনে রাখো |
|---|---|---|
| `np.sqrt(np.sum(a - b) ** 2)` লিখেছি (আগে যোগ, পরে বর্গ) | `np.sqrt(np.sum((a - b) ** 2))` | বর্গ আগে, যোগ পরে, √ সবার শেষে। a = [1, 1], b = [4, 5]-এ ভুল ফল 7.0, ঠিক ফল 5.0 |
| `dot / norm(A) * norm(B)` লিখেছি | `dot / (norm(A) * norm(B))` | ভাগের নিচের অংশ পুরোটা ব্র্যাকেটে |
| `np.linalg.norm(A ** 2)` লিখেছি | `np.linalg.norm(A)` | `norm` নিজেই বর্গ করে, তাই `** 2` লাগে না |

নিচের তিনটা cell প্রতিটা ভুলের ঠিক ফল `assert` দিয়ে যাচাই করে।

In [43]:
a = np.array([1, 1])
b = np.array([4, 5])

wrong = np.sqrt(np.sum(a - b) ** 2)
right = np.sqrt(np.sum((a - b) ** 2))
print("ভুল:", wrong, "| ঠিক:", right)

assert wrong == 7.0
assert right == 5.0


ভুল: 7.0 | ঠিক: 5.0


In [44]:
A = np.array([1, 2])
B = np.array([3, 4])
dot = np.dot(A, B)

wrong = dot / np.linalg.norm(A) * np.linalg.norm(B)
right = dot / (np.linalg.norm(A) * np.linalg.norm(B))
print("ভুল:", round(wrong, 3), "| ঠিক:", round(right, 5))

assert np.isclose(right, 0.98387, atol=1e-5)
assert wrong > 1


ভুল: 24.597 | ঠিক: 0.98387


In [45]:
A = np.array([3, 4])

wrong = np.linalg.norm(A ** 2)
right = np.linalg.norm(A)
print("ভুল:", round(wrong, 3), "| ঠিক:", right)

assert right == 5.0
assert not np.isclose(wrong, 5.0)


ভুল: 18.358 | ঠিক: 5.0


#### Week-end Gate

নিজে চেষ্টা করে তারপর সমাধান মেলাও।

**প্রশ্ন:** `compare(A, B)` নামে একটা function লেখো, যেটা দুটো Vector নিয়ে `(dot, distance, cosine)` ফেরত দেয়। A = [1, 2], B = [3, 4]-এ ফল হবে (11, 2.8284..., 0.9838...)।

<details>
<summary>সমাধান দেখতে click করো</summary>

তিনটা মান এভাবে বের হয়:

> dot = 1×3 + 2×4 = 11
> distance = √[(1 − 3)² + (2 − 4)²] = √(4 + 4) = √8 ≈ 2.8284
> cosine = 11 ÷ (√5 × 5) ≈ 0.98387

```python
def compare(A, B):
    A = np.array(A)
    B = np.array(B)

    dot = np.dot(A, B)
    distance = np.sqrt(np.sum((A - B) ** 2))
    cosine = dot / (np.linalg.norm(A) * np.linalg.norm(B))

    return dot, distance, cosine
```

</details>

In [46]:
def compare(A, B):

    A = np.array(A)
    B = np.array(B)

    dot = np.dot(A, B)
    distance= np.sqrt(np.sum((A - B)**2))
    cosine = dot/(np.sqrt(np.sum(A ** 2)) * np.sqrt(np.sum(B ** 2)))


    return dot, distance, cosine

result = compare([1, 2], [3, 4])
print(result)


(np.int64(11), np.float64(2.8284271247461903), np.float64(0.9838699100999074))


এবার ছোট কয়েকটা প্রশ্ন। নিজে উত্তর ভেবে তারপর মেলাও।

**প্রশ্ন ১:** Dot product কী মাপে?

**প্রশ্ন ২:** Cosine similarity ১ হলে কী বোঝায়?

**প্রশ্ন ৩:** Euclidean distance আর cosine similarity-র পার্থক্য কী?

**প্রশ্ন ৪:** Unit vector-এর দৈর্ঘ্য কত?

<details>
<summary>উত্তর দেখতে click করো</summary>

১. দুই Vector-এর একই জায়গার সংখ্যা গুণ করে যোগ করলে যে সংখ্যা পাওয়া যায়, সেটাই dot product। এটা দুই Vector কতটা মেলে, তার একটা মাপ। তবে Vector বড় হলে ফল এমনিতেই বড় হয়।

২. **দুই Vector একই দিকে**, অর্থাৎ অর্থ খুব কাছাকাছি।

৩. Euclidean distance দুটো বিন্দু কতটা **দূরে**, তা মাপে, আর কম হলে কাছাকাছি। Cosine similarity দুটো Vector-এর **দিক** কতটা মেলে, তা মাপে, আর বেশি হলে কাছাকাছি। যেমন [1, 1] আর [10, 10]-এর দূরত্ব অনেক, কিন্তু দিক একই বলে cosine ১।

৪. **১।**

</details>

নিচের cell-এ উত্তরগুলো NumPy দিয়ে মিলিয়ে দেখা যায়।

In [47]:
print(compare([1, 2], [3, 4]))
assert np.allclose(compare([1, 2], [3, 4]), (11, 2.8284271247, 0.9838699101))

p = np.array([1, 1])
q = np.array([10, 10])
print(round(np.sqrt(np.sum((p - q) ** 2)), 3))
print(round(cosine(p, q), 3))

unit = np.array([3, 4]) / np.linalg.norm([3, 4])
print(np.linalg.norm(unit))


(np.int64(11), np.float64(2.8284271247461903), np.float64(0.9838699100999074))
12.728
1.0
1.0
